# Reporte Tecnico Qwen3 Evaluation

Notebook standalone para leer los outputs de evaluacion del LLM Qwen3, derivar metricas tecnicas adicionales y generar un HTML autocontenido para diagnostico.

Ruta objetivo por defecto:

```text
outputs/sft_llm_qwen3/evaluation/
```

Artefactos esperados:

- `qwen3_text_llm_metrics.csv`
- `qwen3_text_finetuned_outputs.json`
- `qwen3_text_base_outputs.json` (opcional)
- `qwen3_text_comparison_table.csv` (opcional)


In [1]:
import json
import re
from datetime import datetime
from html import escape
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display


PROJECT_ROOT = Path("..").resolve()
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
TECHNICAL_EVAL_DIR = OUTPUT_ROOT / "sft_llm_qwen3" / "evaluation"
TECHNICAL_REPORT_PATH = TECHNICAL_EVAL_DIR / "qwen3_text_technical_report.html"


In [2]:
tech_paths = {
    "metrics": TECHNICAL_EVAL_DIR / "qwen3_text_llm_metrics.csv",
    "comparison": TECHNICAL_EVAL_DIR / "qwen3_text_comparison_table.csv",
    "fine_tuned": TECHNICAL_EVAL_DIR / "qwen3_text_finetuned_outputs.json",
    "base": TECHNICAL_EVAL_DIR / "qwen3_text_base_outputs.json",
}

required_paths = [tech_paths["metrics"], tech_paths["fine_tuned"]]
missing_required = [str(path) for path in required_paths if not path.exists()]
if missing_required:
    raise FileNotFoundError(
        "Faltan archivos requeridos para el reporte tecnico: " + ", ".join(missing_required)
    )

for label, path in tech_paths.items():
    print(label, "->", path.exists(), path)

metrics_df = pd.read_csv(tech_paths["metrics"])
fine_tuned_outputs = json.loads(tech_paths["fine_tuned"].read_text(encoding="utf-8"))
comparison_df = pd.read_csv(tech_paths["comparison"]) if tech_paths["comparison"].exists() else pd.DataFrame()
base_outputs = json.loads(tech_paths["base"].read_text(encoding="utf-8")) if tech_paths["base"].exists() else []

print("metrics rows:", len(metrics_df))
print("fine_tuned outputs:", len(fine_tuned_outputs))
print("base outputs:", len(base_outputs))
print("comparison rows:", len(comparison_df))


metrics -> True /Users/chperezpelaez/Documents/Github/dmc-tp3-gen-multimodal/outputs/sft_llm_qwen3/evaluation/qwen3_text_llm_metrics.csv
comparison -> True /Users/chperezpelaez/Documents/Github/dmc-tp3-gen-multimodal/outputs/sft_llm_qwen3/evaluation/qwen3_text_comparison_table.csv
fine_tuned -> True /Users/chperezpelaez/Documents/Github/dmc-tp3-gen-multimodal/outputs/sft_llm_qwen3/evaluation/qwen3_text_finetuned_outputs.json
base -> True /Users/chperezpelaez/Documents/Github/dmc-tp3-gen-multimodal/outputs/sft_llm_qwen3/evaluation/qwen3_text_base_outputs.json
metrics rows: 63
fine_tuned outputs: 60
base outputs: 3
comparison rows: 60


In [3]:
EXPECTED_SFT_OUTPUT_FIELDS = {
    "strategy",
    "recommended_channel",
    "channel_rationale",
    "channel_plan",
    "ad_copy",
    "image_prompt",
    "kpis",
    "business_note",
}

json_decoder = json.JSONDecoder()
PROMPT_ECHO_MARKERS = [
    "### instruction",
    "### input",
    "datos de entrada",
    "answer:",
    "respuesta:",
    "input:",
]
INSTRUCTION_MARKERS = [
    "actua como",
    "responde exclusivamente",
    "respond exclusively",
    "only respond with",
    "valid json",
    "exactamente estos campos",
]

def extract_json_object(text):
    if isinstance(text, dict):
        return text
    if not isinstance(text, str):
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

def try_raw_json_decode(text):
    if not isinstance(text, str):
        return None, None, None
    start = text.find("{")
    if start < 0:
        return None, None, None
    try:
        parsed, end = json_decoder.raw_decode(text[start:])
        return parsed, start, start + end
    except Exception:
        return None, start, None

def build_output_diagnostics(item):
    generated = str(item.get("generated", "") or "")
    stripped = generated.strip()
    lowered = stripped.lower()
    parsed = extract_json_object(generated)
    partial_parsed, _, json_end = try_raw_json_decode(generated)
    parsed_dict = parsed if isinstance(parsed, dict) else {}
    parsed_keys = set(parsed_dict.keys()) if parsed_dict else set()
    missing_keys = sorted(EXPECTED_SFT_OUTPUT_FIELDS - parsed_keys) if parsed_dict else sorted(EXPECTED_SFT_OUTPUT_FIELDS)
    unexpected_keys = sorted(parsed_keys - EXPECTED_SFT_OUTPUT_FIELDS) if parsed_dict else []
    expected = item.get("expected", {}) or {}
    expected_channel = str(expected.get("recommended_channel", "")).strip().lower()
    generated_channel = str(parsed_dict.get("recommended_channel", "")).strip().lower()
    prompt_echo = any(marker in lowered for marker in PROMPT_ECHO_MARKERS)
    extra_instructions = any(marker in lowered for marker in INSTRUCTION_MARKERS)
    trailing_segment = generated[json_end:].strip() if json_end is not None else ""
    extra_after_json = bool(trailing_segment and re.search(r"[A-Za-zÁ-ÿ#]", trailing_segment))
    truncated_json = bool(parsed_dict and missing_keys)
    if not parsed_dict and stripped:
        if not stripped.endswith("}"):
            truncated_json = True
        if re.search(r'[:,\[{(]\s*$', stripped):
            truncated_json = True
        if stripped.count("{") > stripped.count("}"):
            truncated_json = True
        if "{" in stripped:
            truncated_json = True
    exact_schema_match = float(bool(parsed_dict) and not missing_keys and not unexpected_keys)
    schema_status = "invalid_json"
    if parsed_dict:
        if not missing_keys and not unexpected_keys:
            schema_status = "exact"
        elif missing_keys and not unexpected_keys:
            schema_status = "missing_keys"
        elif unexpected_keys and not missing_keys:
            schema_status = "extra_keys"
        else:
            schema_status = "missing_and_extra"
    flags = []
    if parsed is None:
        flags.append("invalid_json")
    if prompt_echo:
        flags.append("prompt_echo")
    if extra_instructions:
        flags.append("extra_instructions")
    if extra_after_json:
        flags.append("extra_after_json")
    if truncated_json:
        flags.append("truncated_json")
    if missing_keys:
        flags.append("schema_incomplete")
    if unexpected_keys:
        flags.append("schema_extra_fields")
    if expected_channel and expected_channel != generated_channel:
        flags.append("channel_mismatch")
    return {
        "example_id": item.get("example_id"),
        "model": item.get("model", "fine_tuned"),
        "generated_len": len(generated),
        "generated_preview": stripped[:420],
        "prompt_echo": float(prompt_echo),
        "extra_instructions": float(extra_instructions),
        "extra_after_json": float(extra_after_json),
        "truncated_json": float(truncated_json),
        "channel_mismatch_flag": float(bool(expected_channel and expected_channel != generated_channel)),
        "missing_keys": missing_keys,
        "unexpected_keys": unexpected_keys,
        "missing_key_count": len(missing_keys),
        "unexpected_key_count": len(unexpected_keys),
        "exact_schema_match": exact_schema_match,
        "schema_status": schema_status,
        "flags": flags,
        "has_partial_json": float(partial_parsed is not None),
    }

ft_diag_df = pd.DataFrame([build_output_diagnostics(item) for item in fine_tuned_outputs])
ft_metrics_df = metrics_df[metrics_df["model"] == "fine_tuned"].copy()
ft_diag_df = ft_diag_df.merge(ft_metrics_df, on=["example_id", "model"], how="left")
ft_diag_df["weak_overlap"] = (ft_diag_df["text_jaccard_vs_expected"].fillna(0) < 0.6).astype(float)
ft_diag_df["weak_kpi_match"] = (ft_diag_df["kpi_jaccard"].fillna(0) < 1.0).astype(float)

def append_metric_flags(row):
    flags = list(row["flags"])
    if row["weak_overlap"]:
        flags.append("weak_overlap")
    if row["weak_kpi_match"]:
        flags.append("weak_kpi_match")
    return sorted(set(flags))

ft_diag_df["flags"] = ft_diag_df.apply(append_metric_flags, axis=1)
ft_diag_df["flag_count"] = ft_diag_df["flags"].apply(len)

model_summary_df = metrics_df.groupby("model", dropna=False).agg(
    examples=("example_id", "count"),
    parse_success_rate=("json_valid", "mean"),
    field_coverage_avg=("field_coverage", "mean"),
    channel_match_rate=("recommended_channel_match", "mean"),
    kpi_jaccard_avg=("kpi_jaccard", "mean"),
    overlap_avg=("text_jaccard_vs_expected", "mean"),
    image_prompt_rate=("image_prompt_present", "mean"),
    latency_avg=("latency_sec", "mean"),
    latency_p90=("latency_sec", lambda s: s.quantile(0.9) if len(s) else 0.0),
    latency_max=("latency_sec", "max"),
).reset_index()

def rate(series):
    return float(series.fillna(0).mean()) if len(series) else 0.0

latency_series = ft_diag_df["latency_sec"].fillna(0)
length_series = ft_diag_df["generated_len"].fillna(0)
kpi_cards = {
    "parse_success_rate": rate(ft_diag_df["json_valid"]),
    "exact_schema_rate": rate(ft_diag_df["exact_schema_match"]),
    "high_quality_rate": rate(
        (
            (ft_diag_df["json_valid"] == 1)
            & (ft_diag_df["field_coverage"] == 1)
            & (ft_diag_df["image_prompt_present"] == 1)
            & (ft_diag_df["text_jaccard_vs_expected"] >= 0.7)
        ).astype(float)
    ),
    "prompt_echo_rate": rate(ft_diag_df["prompt_echo"]),
    "suspicious_truncation_rate": rate(ft_diag_df["truncated_json"]),
    "trailing_garbage_rate": rate(ft_diag_df["extra_after_json"]),
    "channel_error_rate": 1.0 - rate(ft_diag_df["recommended_channel_match"]),
    "kpi_full_match_rate": rate((ft_diag_df["kpi_jaccard"] == 1).astype(float)),
    "schema_incomplete_rate": rate((ft_diag_df["missing_key_count"] > 0).astype(float)),
    "schema_extra_fields_rate": rate((ft_diag_df["unexpected_key_count"] > 0).astype(float)),
}

latency_stats = {
    "min": float(latency_series.min()) if len(latency_series) else 0.0,
    "p50": float(latency_series.quantile(0.5)) if len(latency_series) else 0.0,
    "p90": float(latency_series.quantile(0.9)) if len(latency_series) else 0.0,
    "max": float(latency_series.max()) if len(latency_series) else 0.0,
    "avg": float(latency_series.mean()) if len(latency_series) else 0.0,
}
length_stats = {
    "min": int(length_series.min()) if len(length_series) else 0,
    "p50": float(length_series.quantile(0.5)) if len(length_series) else 0.0,
    "p90": float(length_series.quantile(0.9)) if len(length_series) else 0.0,
    "max": int(length_series.max()) if len(length_series) else 0,
    "avg": float(length_series.mean()) if len(length_series) else 0.0,
}

flag_counts = (
    ft_diag_df.explode("flags")
    .dropna(subset=["flags"])
    .groupby("flags")
    .size()
    .sort_values(ascending=False)
)
flag_summary_df = pd.DataFrame({"flag": flag_counts.index, "count": flag_counts.values}) if len(flag_counts) else pd.DataFrame(columns=["flag", "count"])
if not flag_summary_df.empty:
    flag_summary_df["rate"] = flag_summary_df["count"] / len(ft_diag_df)

problematic_examples_df = ft_diag_df[ft_diag_df["flag_count"] > 0].copy()
problematic_examples_df = problematic_examples_df.sort_values(["flag_count", "json_valid", "latency_sec"], ascending=[False, True, False]).head(12)
best_examples_df = ft_diag_df[(ft_diag_df["json_valid"] == 1) & (ft_diag_df["field_coverage"] == 1) & (ft_diag_df["text_jaccard_vs_expected"] >= 0.8)].copy()
best_examples_df = best_examples_df.sort_values(["text_jaccard_vs_expected", "latency_sec"], ascending=[False, True]).head(8)

display(model_summary_df)
display(flag_summary_df)
display(problematic_examples_df[["example_id", "flags", "missing_keys", "unexpected_keys", "latency_sec", "generated_preview"]].head(10))


,model,examples,parse_success_rate,field_coverage_avg,channel_match_rate,kpi_jaccard_avg,overlap_avg,image_prompt_rate,latency_avg,latency_p90,latency_max
0,base,3,0.0,0.0,0.000000,0.000000,0.000000,0.0,24.504535,34.994939,39.427974
1,fine_tuned,60,0.9,0.9,0.766667,0.853373,0.675422,0.9,95.722466,104.391924,136.350442


,flag,count,rate
0,extra_after_json,43,0.716667
1,extra_instructions,35,0.583333
2,prompt_echo,34,0.566667
3,channel_mismatch,14,0.233333
4,weak_overlap,14,0.233333
5,weak_kpi_match,11,0.183333
6,invalid_json,6,0.100000
7,schema_incomplete,6,0.100000
8,truncated_json,6,0.100000


,example_id,flags,missing_keys,unexpected_keys,latency_sec,generated_preview
59,59,"[channel_mismatch, extra_after_json, extra_ins...","[ad_copy, business_note, channel_plan, channel...",[],98.142980,"{""strategy"": ""Posicionar el Toyota Raize 2024 ..."
0,0,"[channel_mismatch, extra_instructions, invalid...","[ad_copy, business_note, channel_plan, channel...",[],136.350442,"{""strategy"": ""Posicionar el Toyota Corolla Cro..."
11,11,"[channel_mismatch, extra_instructions, invalid...","[ad_copy, business_note, channel_plan, channel...",[],103.334228,"{""strategy"": ""Posicionar el Toyota Raize 2024 ..."
58,58,"[channel_mismatch, extra_instructions, invalid...","[ad_copy, business_note, channel_plan, channel...",[],98.538893,"{""strategy"": ""Posicionar el Toyota Land Cruise..."
26,26,"[channel_mismatch, extra_instructions, invalid...","[ad_copy, business_note, channel_plan, channel...",[],101.067494,"{""strategy"": ""Posicionar el Toyota Corolla Cro..."
37,37,"[channel_mismatch, extra_instructions, invalid...","[ad_copy, business_note, channel_plan, channel...",[],100.570568,"{""strategy"": ""Posicionar el Toyota Yaris Cross..."
21,21,"[channel_mismatch, extra_after_json, extra_ins...",[],[],101.502218,"{""strategy"": ""Posicionar el Toyota Hilux SRV 4..."
39,39,"[channel_mismatch, extra_after_json, extra_ins...",[],[],100.236046,"{""strategy"": ""Posicionar el Toyota Fortuner 20..."
31,31,"[channel_mismatch, extra_after_json, extra_ins...",[],[],99.986024,"{""strategy"": ""Posicionar el Toyota Fortuner 20..."
51,51,"[channel_mismatch, extra_after_json, extra_ins...",[],[],99.131773,"{""strategy"": ""Position Toyota Corolla 2025 aro..."


In [4]:
def fmt_pct(value):
    return f"{value * 100:.1f}%"

def fmt_num(value, digits=3):
    if pd.isna(value):
        return "-"
    return f"{float(value):.{digits}f}"

def fmt_seconds(value):
    if pd.isna(value):
        return "-"
    return f"{float(value):.2f}s"

def truncate_text(value, limit=220):
    text = str(value or "").replace("\n", " ").strip()
    if len(text) <= limit:
        return text
    return text[: limit - 1] + "…"

def html_table(df, columns, formatters=None, max_rows=None, row_class_fn=None):
    formatters = formatters or {}
    working_df = df.head(max_rows) if max_rows else df
    if working_df.empty:
        return '<p class="muted">Sin datos para mostrar.</p>'
    header_html = "".join(f"<th>{escape(str(col))}</th>" for col in columns)
    body_rows = []
    for _, row in working_df.iterrows():
        cell_html = []
        for col in columns:
            formatter = formatters.get(col, lambda x: x)
            value = formatter(row[col])
            cell_html.append(f"<td>{escape(str(value))}</td>")
        row_class = row_class_fn(row) if row_class_fn else ""
        class_attr = f' class="{row_class}"' if row_class else ""
        body_rows.append(f"<tr{class_attr}>" + "".join(cell_html) + "</tr>")
    return "<table><thead><tr>" + header_html + "</tr></thead><tbody>" + "".join(body_rows) + "</tbody></table>"

def bar_rows_html(items, label_fn, value_fn, color="#2563eb"):
    rows = []
    for item in items:
        label = escape(str(label_fn(item)))
        value = max(0.0, min(1.0, float(value_fn(item))))
        rows.append(
            '<div class="bar-row">'
            f'<div class="bar-label">{label}</div>'
            '<div class="bar-track">'
            f'<div class="bar-fill" style="width:{value * 100:.1f}%; background:{color};"></div>'
            '</div>'
            f'<div class="bar-value">{fmt_pct(value)}</div>'
            '</div>'
        )
    return "".join(rows) if rows else '<p class="muted">Sin datos.</p>'

def histogram_svg(values, bins=8, width=460, height=180, color="#0f766e"):
    clean = [float(v) for v in values if pd.notna(v)]
    if not clean:
        return '<p class="muted">Sin datos.</p>'
    vmin, vmax = min(clean), max(clean)
    if vmin == vmax:
        vmax = vmin + 1.0
    step = (vmax - vmin) / bins
    counts = [0] * bins
    for value in clean:
        idx = min(bins - 1, int((value - vmin) / step))
        counts[idx] += 1
    max_count = max(counts) or 1
    bar_width = width / bins
    rects = []
    labels = []
    for idx, count in enumerate(counts):
        bar_height = (count / max_count) * (height - 28)
        x = idx * bar_width + 4
        y = height - bar_height - 22
        rects.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width - 8:.1f}" height="{bar_height:.1f}" rx="4" fill="{color}"></rect>')
        bin_label = f"{vmin + idx * step:.1f}"
        labels.append(f'<text x="{x + (bar_width - 8) / 2:.1f}" y="{height - 6}" text-anchor="middle" font-size="10" fill="#475569">{escape(bin_label)}</text>')
    return f'<svg viewBox="0 0 {width} {height}" class="chart-svg">' + ''.join(rects) + ''.join(labels) + '</svg>'

base_latency = model_summary_df.loc[model_summary_df["model"] == "base", "latency_avg"]
fine_tuned_latency = model_summary_df.loc[model_summary_df["model"] == "fine_tuned", "latency_avg"]
conclusions = []
if kpi_cards["schema_incomplete_rate"] >= 0.25:
    conclusions.append("Una parte importante de las respuestas no completa las 8 llaves esperadas; el principal cuello de botella es el contrato de esquema.")
if kpi_cards["schema_extra_fields_rate"] >= 0.15:
    conclusions.append("Aparecen llaves fuera del contrato esperado; hay evidencia de sobre-generacion o deriva del formato objetivo.")
if kpi_cards["prompt_echo_rate"] >= 0.25:
    conclusions.append("El principal problema es contaminacion del prompt: una fraccion relevante de respuestas reutiliza instrucciones o fragmentos del input.")
if kpi_cards["parse_success_rate"] < 0.85:
    conclusions.append("La mayor perdida de calidad viene por formato JSON invalido o estructura incompleta, no necesariamente por contenido semantico pobre.")
if kpi_cards["high_quality_rate"] >= 0.5:
    conclusions.append("Aun con fallas de formato, ya existe una base de respuestas utiles y cercanas al objetivo esperado.")
if len(base_latency) and len(fine_tuned_latency) and float(fine_tuned_latency.iloc[0]) > float(base_latency.iloc[0]) * 1.5:
    conclusions.append("La latencia fine-tuned es significativamente mayor que baseline; conviene reportarla por separado para no confundir calidad con costo operativo.")
if not conclusions:
    conclusions.append("El run no muestra un patron dominante de fallo; conviene revisar ejemplos concretos y contrastar con la configuracion de generacion.")

flag_items = flag_summary_df.to_dict("records") if not flag_summary_df.empty else []
problem_columns = ["example_id", "schema_status", "missing_keys", "unexpected_keys", "flags", "json_valid", "text_jaccard_vs_expected", "latency_sec", "generated_preview"]
problem_formatters = {
    "schema_status": lambda value: value,
    "missing_keys": lambda value: ", ".join(value) if isinstance(value, list) else value,
    "unexpected_keys": lambda value: ", ".join(value) if isinstance(value, list) else value,
    "flags": lambda value: ", ".join(value) if isinstance(value, list) else value,
    "json_valid": lambda value: "yes" if float(value) == 1.0 else "no",
    "text_jaccard_vs_expected": lambda value: fmt_num(value, 3),
    "latency_sec": fmt_seconds,
    "generated_preview": lambda value: truncate_text(value, 240),
}
best_columns = ["example_id", "text_jaccard_vs_expected", "kpi_jaccard", "latency_sec", "generated_preview"]
best_formatters = {
    "text_jaccard_vs_expected": lambda value: fmt_num(value, 3),
    "kpi_jaccard": lambda value: fmt_num(value, 3),
    "latency_sec": fmt_seconds,
    "generated_preview": lambda value: truncate_text(value, 240),
}

comparison_section_html = '<p class="muted">No se encontro comparacion base vs fine-tuned para este run.</p>'
if not comparison_df.empty:
    comparison_preview = comparison_df.copy()
    comparison_preview["input"] = comparison_preview["input"].map(lambda value: truncate_text(value, 140))
    comparison_preview["base_generated"] = comparison_preview["base_generated"].map(lambda value: truncate_text(value, 160))
    comparison_preview["fine_tuned_generated"] = comparison_preview["fine_tuned_generated"].map(lambda value: truncate_text(value, 160))
    comparison_section_html = html_table(comparison_preview, ["example_id", "input", "base_generated", "fine_tuned_generated"], max_rows=6)

summary_table_html = html_table(
    model_summary_df,
    ["model", "examples", "parse_success_rate", "field_coverage_avg", "channel_match_rate", "kpi_jaccard_avg", "overlap_avg", "image_prompt_rate", "latency_avg", "latency_p90", "latency_max"],
    formatters={
        "parse_success_rate": fmt_pct,
        "field_coverage_avg": fmt_pct,
        "channel_match_rate": fmt_pct,
        "kpi_jaccard_avg": lambda value: fmt_num(value, 3),
        "overlap_avg": lambda value: fmt_num(value, 3),
        "image_prompt_rate": fmt_pct,
        "latency_avg": fmt_seconds,
        "latency_p90": fmt_seconds,
        "latency_max": fmt_seconds,
    },
)

html_report = f"""
<!DOCTYPE html>
<html lang=\"es\">
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <title>Qwen3 Technical Evaluation Report</title>
  <style>
    :root {{
      color-scheme: light;
      --bg: #f8fafc;
      --panel: #ffffff;
      --ink: #0f172a;
      --muted: #475569;
      --line: #dbe4ee;
      --accent: #2563eb;
      --warn: #b91c1c;
      --good: #0f766e;
    }}
    body {{ margin: 0; background: var(--bg); color: var(--ink); font-family: Arial, sans-serif; }}
    .wrap {{ max-width: 1200px; margin: 0 auto; padding: 32px 24px 64px; }}
    h1, h2, h3 {{ margin: 0 0 12px; }}
    h1 {{ font-size: 30px; }}
    h2 {{ font-size: 22px; margin-top: 34px; }}
    p {{ color: var(--muted); line-height: 1.5; }}
    .meta {{ display: flex; flex-wrap: wrap; gap: 16px; margin: 14px 0 28px; color: var(--muted); font-size: 14px; }}
    .cards {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 14px; }}
    .card, .panel {{ background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 16px; box-shadow: 0 1px 2px rgba(15, 23, 42, 0.04); }}
    .card .label {{ color: var(--muted); font-size: 13px; margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.04em; }}
    .card .value {{ font-size: 28px; font-weight: 700; margin-bottom: 4px; }}
    .grid-2 {{ display: grid; grid-template-columns: 1.2fr 1fr; gap: 16px; align-items: start; }}
    .grid-2.equal {{ grid-template-columns: 1fr 1fr; }}
    table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
    th, td {{ border-bottom: 1px solid var(--line); padding: 10px 8px; text-align: left; vertical-align: top; }}
    th {{ color: var(--muted); font-weight: 600; background: #f8fafc; }}
    tr.bad-row td {{ background: #fff1f2; }}
    .muted {{ color: var(--muted); }}
    .bar-row {{ display: grid; grid-template-columns: 170px 1fr 72px; gap: 10px; align-items: center; margin: 10px 0; }}
    .bar-label, .bar-value {{ font-size: 13px; color: var(--muted); }}
    .bar-track {{ background: #e2e8f0; border-radius: 999px; height: 12px; overflow: hidden; }}
    .bar-fill {{ height: 100%; border-radius: 999px; }}
    .chart-svg {{ width: 100%; height: auto; display: block; }}
    ul.conclusions {{ margin: 0; padding-left: 18px; color: var(--muted); }}
    code {{ background: #eff6ff; padding: 2px 5px; border-radius: 4px; }}
    @media (max-width: 900px) {{
      .grid-2, .grid-2.equal {{ grid-template-columns: 1fr; }}
      .bar-row {{ grid-template-columns: 1fr; }}
    }}
  </style>
</head>
<body>
  <div class=\"wrap\">
    <h1>Qwen3 Technical Evaluation Report</h1>
    <p>Diagnostico tecnico sobre outputs fine-tuned y metricas agregadas del run almacenado en <code>{escape(str(TECHNICAL_EVAL_DIR))}</code>.</p>
    <div class=\"meta\">
      <div><strong>Generado:</strong> {escape(datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z'))}</div>
      <div><strong>Ejemplos fine-tuned:</strong> {len(ft_diag_df)}</div>
      <div><strong>Base disponible:</strong> {'si' if base_outputs else 'no'}</div>
      <div><strong>HTML:</strong> {escape(str(TECHNICAL_REPORT_PATH))}</div>
    </div>

    <div class=\"cards\">
      <div class=\"card\"><div class=\"label\">Parse success</div><div class=\"value\">{fmt_pct(kpi_cards['parse_success_rate'])}</div><div class=\"muted\">JSON parseable</div></div>
      <div class=\"card\"><div class=\"label\">Exact schema</div><div class=\"value\">{fmt_pct(kpi_cards['exact_schema_rate'])}</div><div class=\"muted\">Las 8 llaves exactas</div></div>
      <div class=\"card\"><div class=\"label\">High quality</div><div class=\"value\">{fmt_pct(kpi_cards['high_quality_rate'])}</div><div class=\"muted\">JSON valido + cobertura + overlap >= 0.7</div></div>
      <div class=\"card\"><div class=\"label\">Prompt echo</div><div class=\"value\">{fmt_pct(kpi_cards['prompt_echo_rate'])}</div><div class=\"muted\">Respuestas con restos del prompt</div></div>
      <div class=\"card\"><div class=\"label\">Schema incomplete</div><div class=\"value\">{fmt_pct(kpi_cards['schema_incomplete_rate'])}</div><div class=\"muted\">Faltan llaves esperadas</div></div>
      <div class=\"card\"><div class=\"label\">Schema extra</div><div class=\"value\">{fmt_pct(kpi_cards['schema_extra_fields_rate'])}</div><div class=\"muted\">Sobran llaves fuera del contrato</div></div>
      <div class=\"card\"><div class=\"label\">Latency avg / p90</div><div class=\"value\">{fmt_seconds(latency_stats['avg'])}</div><div class=\"muted\">p90 {fmt_seconds(latency_stats['p90'])}</div></div>
    </div>

    <h2>Metricas agregadas por modelo</h2>
    <div class=\"panel\">{summary_table_html}</div>

    <div class=\"grid-2\">
      <div class=\"panel\">
        <h2>Patrones de fallo</h2>
        {bar_rows_html(flag_items, lambda item: item['flag'], lambda item: item['rate'], color='#b91c1c')}
      </div>
      <div class=\"panel\">
        <h2>Conclusiones automaticas</h2>
        <ul class=\"conclusions\">{''.join(f'<li>{escape(item)}</li>' for item in conclusions)}</ul>
        <h3 style=\"margin-top:20px;\">Longitud de salida</h3>
        <p class=\"muted\">min {length_stats['min']} | p50 {length_stats['p50']:.1f} | p90 {length_stats['p90']:.1f} | max {length_stats['max']} | avg {length_stats['avg']:.1f}</p>
      </div>
    </div>

    <div class=\"grid-2 equal\">
      <div class=\"panel\">
        <h2>Distribucion de latencia</h2>
        {histogram_svg(latency_series.tolist(), bins=8, color='#2563eb')}
      </div>
      <div class=\"panel\">
        <h2>Distribucion de overlap</h2>
        {histogram_svg(ft_diag_df['text_jaccard_vs_expected'].fillna(0).tolist(), bins=8, color='#0f766e')}
      </div>
    </div>

    <h2>Ejemplos problematicos</h2>
    <div class=\"panel\">{html_table(problematic_examples_df, problem_columns, formatters=problem_formatters, row_class_fn=lambda row: 'bad-row' if float(row['json_valid']) == 0 else '')}</div>

    <h2>Mejores respuestas</h2>
    <div class=\"panel\">{html_table(best_examples_df, best_columns, formatters=best_formatters)}</div>

    <h2>Comparacion base vs fine-tuned</h2>
    <div class=\"panel\">{comparison_section_html}</div>
  </div>
</body>
</html>
"""

TECHNICAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)
TECHNICAL_REPORT_PATH.write_text(html_report, encoding="utf-8")
print("Reporte tecnico HTML guardado en:", TECHNICAL_REPORT_PATH)
display(HTML('<p><strong>Reporte tecnico generado.</strong> Abre el archivo HTML en la carpeta de evaluacion para ver el diagnostico completo.</p>'))


Reporte tecnico HTML guardado en: /Users/chperezpelaez/Documents/Github/dmc-tp3-gen-multimodal/outputs/sft_llm_qwen3/evaluation/qwen3_text_technical_report.html
